In [1]:
!pip install -q langchain langchain-community langchain-google-genai langchain-text-splitters faiss-cpu pypdf

In [2]:
from google.colab import files

uploaded = files.upload()

Saving Alphabet 10K 2024.pdf to Alphabet 10K 2024 (1).pdf
Saving Amazon 10K 2024.pdf to Amazon 10K 2024 (1).pdf
Saving MSFT 10-K.pdf to MSFT 10-K (1).pdf


In [3]:
from langchain_community.document_loaders import PyPDFLoader

docs = []

files = [
    "Alphabet 10K 2024.pdf",
    "Amazon 10K 2024.pdf",
    "MSFT 10-K.pdf"
]

for file in files:
    loader = PyPDFLoader(file)
    docs.extend(loader.load())

print("Total pages loaded:", len(docs))

Total pages loaded: 409


In [4]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter new Gemini API key: ")
print("API key loaded")

Enter new Gemini API key: ··········
API key loaded


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

documents = text_splitter.split_documents(docs)

print("Total chunks:", len(documents))

Total chunks: 868


In [6]:
from langchain_community.document_loaders import PyPDFLoader

company_docs = {}

file_company_map = {
    "Alphabet 10K 2024.pdf": "Alphabet",
    "Amazon 10K 2024.pdf": "Amazon",
    "MSFT 10-K.pdf": "Microsoft"
}

for file, company in file_company_map.items():
    loader = PyPDFLoader(file)
    loaded_docs = loader.load()
    for d in loaded_docs:
        d.metadata["company"] = company
        d.metadata["source_file"] = file
    company_docs[company] = loaded_docs

for company in company_docs:
    print(company, len(company_docs[company]))

Alphabet 99
Amazon 89
Microsoft 221


In [7]:
import time
from langchain_community.vectorstores import FAISS

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001")

batch_size = 100
vectorstore = None

for i in range(0, len(documents), batch_size):
    batch = documents[i:i+batch_size]
    if vectorstore is None:
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        vectorstore.add_documents(batch)

print("Vector DB created with all documents")

Vector DB created with all documents


In [8]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

print("Retriever ready")

Retriever ready


Group5 Question


In [9]:
# Group 5: Regulatory Risks Comparison
query = "Using only the information in the latest 10-K filings of Alphabet, Amazon, and Microsoft, identify three regulatory risks they all face."

# Check if required objects exist before running
if 'retriever' not in globals() or 'llm' not in globals():
    print("CRITICAL ERROR: 'retriever' or 'llm' is not defined. Please run the setup cells above first.")
else:
    # 1. Retrieve the relevant document chunks
    docs_found = retriever.invoke(query) #cite: 261

    # 2. Build the context string safely
    context_list = []
    for doc in docs_found:
        # Using .get() to prevent KeyError if metadata is missing
        company = doc.metadata.get('company', 'Unknown Company')
        source = doc.metadata.get('source_file', doc.metadata.get('source', 'Unknown File'))
        page = doc.metadata.get('page', 'N/A')

        header = f"--- Company: {company} | Source: {source} (Page {page}) ---"
        context_list.append(f"{header}\n{doc.page_content}")

    context = "\n\n".join(context_list) #cite: 262-266

    # 3. Define the Prompt
    prompt_text = f"""
    You are a financial analysis assistant.
    Answer the question only based on the provided context from the uploaded 10-K filings.
    When comparing companies, be explicit and name the company.
    Identify three regulatory risks (e.g., AI, Antitrust, Privacy) common to Alphabet, Amazon, and Microsoft.

    Context:
    {context}

    Question:
    {query}
    """

    # 4. Generate response
    print(f"Successfully retrieved {len(docs_found)} document chunks.")
    response = llm.invoke(prompt_text)
    print("\n" + "="*20 + " ANALYSIS RESULT " + "="*20 + "\n")
    print(response.content)

CRITICAL ERROR: 'retriever' or 'llm' is not defined. Please run the setup cells above first.


In [10]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

print("Retriever ready")

Retriever ready


In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("LLM ready")

LLM ready


In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a financial analysis assistant.

Answer the question only based on the provided context.
If the answer is not in the context, say:
"I cannot find the answer in the provided documents."

Context:
{context}

Question:
{input}
""")

print("Prompt ready")

Prompt ready


In [13]:
query = "What are the main businesses of Amazon?"

docs_found = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in docs_found])

print("Retrieved docs:", len(docs_found))
print(context[:2000])

Retrieved docs: 6
TV, Echo, Ring, Blink, and eero, and we develop and produce media content. We seek to offer our customers low prices, fast and free delivery, easy-to-use
functionality, and timely customer service. In addition, we offer subscription services such as Amazon Prime, a membership program that includes fast, free
shipping on tens of millions of items, access to award-winning movies and series, live sports, and other benefits.
We fulfill customer orders in a number of ways, including through: North America and International fulfillment networks that we operate; co-sourced and
outsourced arrangements in certain countries; digital delivery; and through our physical stores. We operate customer service centers globally, which are
supplemented by co-sourced arrangements. See Item 2 of Part I, “Properties.”
Sellers
We offer programs that enable sellers to grow their businesses, sell their products in our stores, and fulfill orders using our services. We are not the seller
of reco

Group 2 Question

In [14]:
query = "Which company reported the fastest growth in its cloud segment between 2023 and 2024, and what factors did they attribute this growth to?"

docs_found = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in docs_found])

print("Retrieved docs:", len(docs_found))
print(context[:3000])

Retrieved docs: 6
PART II
Item 7
 
SEGMENT RESULTS OF OPERATIONS
 
(In millions, except percentages)  2024  2023  
Percentage
Change  
          
          
Revenue             
    
Productivity and Business Processes  $ 77,728  $ 69,274    12% 
Intelligent Cloud   105,362   87,907    20% 
More Personal Computing   62,032   54,734    13% 
          
          
Total  $ 245,122  $ 211,915    16% 
             
    
Operating Income             
             
Productivity and Business Processes  $ 40,540  $ 34,189    19% 
Intelligent Cloud   49,584   37,884    31% 
More Personal Computing   19,309   16,450    17% 
           
          
Total  $ 109,433  $ 88,523    24% 
             
 
Reportable Segments
Fiscal Year 2024 Compared with Fiscal Year 2023 
Productivity and Business Processes 
Revenue increased $8.5 billion or 12%.
• Office Commercial products and cloud services revenue increased $5.8 billion or 14%. Office 365 Commercial 
revenue grew 16% with seat growth of 7%, driven by

In [15]:
prompt = f"""
You are a financial analysis assistant.

Answer the question only based on the provided context from the uploaded 10-K filings.
When comparing companies, be explicit and name the company.
If percentages or growth rates are mentioned, include them exactly as stated in the context.
If the answer is not clearly supported by the context, say so instead of guessing.

Context:
{context}

Question:
{query}
"""

In [16]:
response = llm.invoke(prompt)
print(response.content)

Google reported the fastest growth in its cloud segment between 2023 and 2024.

**Google's** Google Cloud revenues increased $10.1 billion from 2023 to 2024. This represents a growth of approximately 30.65% (calculated as ($43,229 million - $33,088 million) / $33,088 million). The growth was primarily driven by growth in Google Cloud Platform largely from infrastructure services.

In comparison, **Microsoft's** Intelligent Cloud segment revenue increased 20% between 2023 and 2024. Within Microsoft's offerings, Server products and cloud services revenue increased 22%, driven by Azure and other cloud services growth of 30%.


Group 1 Question

In [17]:
query = "Which company appears to have the strongest enterprise AI infrastructure advantage?"

docs_found = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in docs_found])

print("Retrieved docs:", len(docs_found))
print(context[:3000])

Retrieved docs: 6
PART I
Item 1
 
Our AI platform, Azure AI, is helping organizations transform, bringing intelligence and insights to the hands of their 
employees and customers to solve their most pressing challenges. We offer a wide selection of industry-leading frontier 
and open models, including from partners, as well as state-of-the-art tooling, and AI-optimized infrastructure, delivering the 
Copilot stack for Microsoft, enterprises, and developers. Organizations large and small are deploying Azure AI solutions to 
achieve more at scale, more easily, with the proper enterprise-level responsible AI and safety and security protections. 
Azure AI Studio provides a full lifecycle toolchain customers can use to ground these models on their own data, create 
prompt workflows, and help ensure they are deployed and used safely.
GitHub Copilot is at the forefront of AI-powered software development, giving developers a tool to write code easier and 
faster. From GitHub to Visual Studio, 

Group 3 Question

In [18]:
query = "According to Amazon’s latest 10-K, what are the main sources of Amazon’s operating income, and which segment contributes the most?"

docs_found = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in docs_found])

print("Retrieved docs:", len(docs_found))
print(context[:3000])

Retrieved docs: 6
Table of Contents
General and Administrative
The decrease in general and administrative costs in 2024, compared to the prior year, is primarily due to a decrease in payroll and related expenses.
Other Operating Expense (Income), Net
Other operating expense (income), net was $767 million and $763 million during 2023 and 2024, and was primarily related to asset impairments and the
amortization of intangible assets.
Operating Income (Loss)
Operating income (loss) by segment is as follows (in millions):
Year Ended December 31,
2023 2024
Operating Income (Loss)
North America $ 14,877 $ 24,967 
International (2,656) 3,792 
AWS 24,631 39,834 
Consolidated $ 36,852 $ 68,593 
Operating income was $36.9 billion and $68.6 billion for 2023 and 2024. We believe that operating income is a more meaningful measure than gross
profit and gross margin due to the diversity of our product categories and services. For more information on the operating expenses that impact segment
operating

Boundary Test

In [19]:
queries = [
    "Which company will dominate cloud computing in 2030?",
    "What will Amazon's AI revenue be in 2026?"
]

for q in queries:

    prompt = f"""
    Answer the following question clearly and concisely.

    Question:
    {q}
    """

    response = llm.invoke(prompt)

    print("Question:", q)
    print("Answer:", response.content)
    print("\n----------------------------\n")

Question: Which company will dominate cloud computing in 2030?
Answer: Predicting a single dominant company in cloud computing by 2030 is challenging due to the rapid evolution of technology and market dynamics. However, the most likely scenario is continued strong competition among the current leaders, rather than absolute dominance by one.

The top contenders expected to lead the market are:

1.  **Amazon Web Services (AWS):** Currently the market leader, with a vast ecosystem, continuous innovation, and strong enterprise adoption.
2.  **Microsoft Azure:** Benefits from Microsoft's extensive enterprise relationships, hybrid cloud capabilities, and strong integration with its software ecosystem.
3.  **Google Cloud Platform (GCP):** A strong third, known for its AI/ML capabilities, data analytics, and open-source contributions, steadily gaining enterprise traction.

It's more probable that these three will continue to be the primary players, with **AWS and Microsoft Azure** likely main

Openai

In [20]:
!pip install langchain-openai

In [21]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
print("OpenAI API key loaded")

Enter your OpenAI API key: ··········
OpenAI API key loaded


In [22]:
from langchain_openai import ChatOpenAI

# Initialize the OpenAI model (using gpt-4o for a fair comparison with gemini-2.5-flash)
llm_openai = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)
print("OpenAI LLM ready")

OpenAI LLM ready


In [23]:
from langchain_core.prompts import ChatPromptTemplate
query = "What are the main businesses of Amazon?"

docs_found = retriever.invoke(query)
def infer_company_from_metadata(doc):
    source = str(doc.metadata.get("source_file") or doc.metadata.get("source") or "").lower()
    if "amazon" in source:
        return "Amazon"
    elif "alphabet" in source or "google" in source:
        return "Alphabet"
    elif "msft" in source or "microsoft" in source:
        return "Microsoft"
    return "Unknown"

context = "\n\n".join([
    f"Company: {infer_company_from_metadata(doc)}\n"
    f"Source: {doc.metadata.get('source_file', doc.metadata.get('source','Unknown'))}\n"
    f"Page: {doc.metadata.get('page','Unknown')}\n"
    f"Content: {doc.page_content}"
    for doc in docs_found
])

print("Retrieved docs:", len(docs_found))
print(context[:3000])


Retrieved docs: 6
Company: Amazon
Source: Amazon 10K 2024.pdf
Page: 2
Content: TV, Echo, Ring, Blink, and eero, and we develop and produce media content. We seek to offer our customers low prices, fast and free delivery, easy-to-use
functionality, and timely customer service. In addition, we offer subscription services such as Amazon Prime, a membership program that includes fast, free
shipping on tens of millions of items, access to award-winning movies and series, live sports, and other benefits.
We fulfill customer orders in a number of ways, including through: North America and International fulfillment networks that we operate; co-sourced and
outsourced arrangements in certain countries; digital delivery; and through our physical stores. We operate customer service centers globally, which are
supplemented by co-sourced arrangements. See Item 2 of Part I, “Properties.”
Sellers
We offer programs that enable sellers to grow their businesses, sell their products in our stores, and ful

In [24]:
prompt_template = ChatPromptTemplate.from_template("""
You are a financial analysis assistant.

Answer the question only based on the provided context from the uploaded 10-K filings.
When comparing companies, explicitly name the company or companies.
If the context does not contain enough evidence, say:
"I cannot find sufficient evidence in the provided documents."

Context:
{context}

Question:
{input}
""")

formatted_prompt = prompt_template.format(
    context=context,
    input=query)

response_openai = llm_openai.invoke(formatted_prompt)

print("OpenAI answer:")
print(response_openai.content)

OpenAI answer:
Amazon's main businesses include:

1. **Retail Sales**: Amazon operates online and physical stores, offering a wide selection of consumer products. This includes both products sold directly by Amazon and those sold by third-party sellers.

2. **Amazon Web Services (AWS)**: AWS provides a broad set of on-demand technology services, including compute, storage, database, analytics, and machine learning, serving developers and enterprises of all sizes.

3. **Subscription Services**: Amazon offers subscription services such as Amazon Prime, which includes benefits like fast, free shipping, access to movies and series, live sports, and other digital content subscriptions.

4. **Advertising Services**: Amazon provides advertising services through programs such as sponsored ads, display, and video advertising.

5. **Content Creation and Distribution**: Amazon develops and produces media content and offers programs for authors, independent publishers, musicians, filmmakers, and o

Group 3 Question(openai)

In [25]:
query = "According to Amazon’s latest 10-K, what are the main sources of Amazon’s operating income, and which segment contributes the most?"

docs_found = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in docs_found])

print("Retrieved docs:", len(docs_found))
print(context[:3000])

Retrieved docs: 6
Table of Contents
General and Administrative
The decrease in general and administrative costs in 2024, compared to the prior year, is primarily due to a decrease in payroll and related expenses.
Other Operating Expense (Income), Net
Other operating expense (income), net was $767 million and $763 million during 2023 and 2024, and was primarily related to asset impairments and the
amortization of intangible assets.
Operating Income (Loss)
Operating income (loss) by segment is as follows (in millions):
Year Ended December 31,
2023 2024
Operating Income (Loss)
North America $ 14,877 $ 24,967 
International (2,656) 3,792 
AWS 24,631 39,834 
Consolidated $ 36,852 $ 68,593 
Operating income was $36.9 billion and $68.6 billion for 2023 and 2024. We believe that operating income is a more meaningful measure than gross
profit and gross margin due to the diversity of our product categories and services. For more information on the operating expenses that impact segment
operating

Group 2 Question(openai)



In [26]:
query = "Which company reported the fastest growth in its cloud segment between 2023 and 2024, and what factors did they attribute this growth to?"

docs_found = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in docs_found])

print("Retrieved docs:", len(docs_found))
print(context[:3000])

Retrieved docs: 6
PART II
Item 7
 
SEGMENT RESULTS OF OPERATIONS
 
(In millions, except percentages)  2024  2023  
Percentage
Change  
          
          
Revenue             
    
Productivity and Business Processes  $ 77,728  $ 69,274    12% 
Intelligent Cloud   105,362   87,907    20% 
More Personal Computing   62,032   54,734    13% 
          
          
Total  $ 245,122  $ 211,915    16% 
             
    
Operating Income             
             
Productivity and Business Processes  $ 40,540  $ 34,189    19% 
Intelligent Cloud   49,584   37,884    31% 
More Personal Computing   19,309   16,450    17% 
           
          
Total  $ 109,433  $ 88,523    24% 
             
 
Reportable Segments
Fiscal Year 2024 Compared with Fiscal Year 2023 
Productivity and Business Processes 
Revenue increased $8.5 billion or 12%.
• Office Commercial products and cloud services revenue increased $5.8 billion or 14%. Office 365 Commercial 
revenue grew 16% with seat growth of 7%, driven by

Group 1 Question

In [27]:
query = "Which company appears to have the strongest enterprise AI infrastructure advantage?"

docs_found = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in docs_found])

print("Retrieved docs:", len(docs_found))
print(context[:3000])

Retrieved docs: 6
PART I
Item 1
 
Our AI platform, Azure AI, is helping organizations transform, bringing intelligence and insights to the hands of their 
employees and customers to solve their most pressing challenges. We offer a wide selection of industry-leading frontier 
and open models, including from partners, as well as state-of-the-art tooling, and AI-optimized infrastructure, delivering the 
Copilot stack for Microsoft, enterprises, and developers. Organizations large and small are deploying Azure AI solutions to 
achieve more at scale, more easily, with the proper enterprise-level responsible AI and safety and security protections. 
Azure AI Studio provides a full lifecycle toolchain customers can use to ground these models on their own data, create 
prompt workflows, and help ensure they are deployed and used safely.
GitHub Copilot is at the forefront of AI-powered software development, giving developers a tool to write code easier and 
faster. From GitHub to Visual Studio, 

Group 5 Question

In [28]:
query = "Using only the information in the latest 10-K filings of Alphabet, Amazon, and Microsoft, identify three regulatory risks they all face."
if 'retriever' not in globals() or 'llm' not in globals():
    print("CRITICAL ERROR: 'retriever' or 'llm' is not defined. Please run the setup cells above first.")
else:
    docs_found = retriever.invoke(query)

    context_list = []
    for doc in docs_found:

        company = doc.metadata.get('company', 'Unknown Company')
        source = doc.metadata.get('source_file', doc.metadata.get('source', 'Unknown File'))
        page = doc.metadata.get('page', 'N/A')

        header = f"--- Company: {company} | Source: {source} (Page {page}) ---"
        context_list.append(f"{header}\n{doc.page_content}")

    context = "\n\n".join(context_list)

    # 3. Define the Prompt (Moved here to ensure it's defined)
    prompt_text = f"""
    You are a financial analysis assistant.
    Answer the question only based on the provided context from the uploaded 10-K filings.
    When comparing companies, be explicit and name the company.
    Identify three regulatory risks (e.g., AI, Antitrust, Privacy) common to Alphabet, Amazon, and Microsoft.

    Context:
    {context}

    Question:
    {query}
    """

    print(f"Successfully retrieved {len(docs_found)} document chunks.")
    response = llm.invoke(prompt_text)
    print("\n" + "="*20 + " ANALYSIS RESULT " + "="*20 + "\n")
    print(response.content)


Successfully retrieved 6 document chunks.

==================== ANALYSIS RESULT ====================

Based on the provided 10-K filings, Alphabet, Amazon, and Microsoft all face regulatory risks related to:

1.  **AI (Artificial Intelligence)**:
    *   **Alphabet** states, "Particularly with regard to AI... we have seen an increase in new and evolving laws and regulations" (Alphabet 10K 2024.pdf, Page 10) and mentions "AI" as an area with a "large number of new laws and regulations" (Alphabet 10K 2024.pdf, Page 18).
    *   **Amazon** notes that "It is not clear how existing laws governing issues such as... artificial intelligence technologies and services... apply to aspects of our operations" (Amazon 10K 2024.pdf, Page 14).
    *   **Microsoft** has established a regulatory governance framework with an initial focus on "Responsible AI" (MSFT 10-K.pdf, Page 27) and states that "Legislative and regulatory action is emerging in the areas of AI" (MSFT 10-K.pdf, Page 46).

2.  **Competi

Boundary test

In [29]:
queries = [
    "Which company will dominate cloud computing in 2030?",
    "What will Amazon's AI revenue be in 2026?"
]

for q in queries:

    prompt = f"""
    Answer the following question clearly and concisely.

    Question:
    {q}
    """

    response = llm_openai.invoke(prompt)

    print("Question:", q)
    print("Answer:", response.content)
    print("\n----------------------------\n")

Question: Which company will dominate cloud computing in 2030?
Answer: Predicting which company will dominate cloud computing in 2030 is speculative and uncertain. As of now, major players in the cloud computing industry include Amazon Web Services (AWS), Microsoft Azure, and Google Cloud Platform. These companies have significant resources, infrastructure, and market share, positioning them well for future growth. However, technological advancements, market dynamics, regulatory changes, and emerging competitors could influence the landscape by 2030. Therefore, it's impossible to definitively state which company will dominate the industry at that time.

----------------------------

Question: What will Amazon's AI revenue be in 2026?
Answer: I'm sorry, but I cannot provide specific future financial projections for Amazon's AI revenue in 2026. Such forecasts would depend on a variety of factors, including market trends, technological advancements, competitive landscape, and strategic de

In [30]:
import ipywidgets as widgets
from IPython.display import display, clear_output

In [31]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown


display(Markdown("## Interactive 10-K Chatbot"))

question_box = widgets.Textarea(
    value="",
    placeholder="Ask a question about the uploaded 10-K filings...",
    description="Question:",
    layout=widgets.Layout(width="90%", height="100px")
)

ask_button = widgets.Button(
    description="Ask",
    button_style="primary",
    icon="search"
)

output = widgets.Output()

def format_docs_for_context(docs):
    context_parts = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source_file", doc.metadata.get("source", "Unknown source"))
        context_parts.append(f"[Document {i} | Source: {source}]\n{doc.page_content}")
    return "\n\n".join(context_parts)

def ask_10k_question(question):
    docs_found = retriever.invoke(question)
    context = format_docs_for_context(docs_found)

    prompt = f"""
You are a financial analysis assistant.

Answer the question only based on the provided context from the uploaded 10-K filings.
When comparing companies, explicitly name the company or companies.
If the context does not contain enough evidence, say:
"I cannot find sufficient evidence in the provided documents."

Question:
{question}

Context:
{context}
"""
    response = llm.invoke(prompt)

    if hasattr(response, "content"):
        return response.content, docs_found
    else:
        return str(response), docs_found

def on_ask_clicked(b):
    with output:
        clear_output()

        question = question_box.value.strip()

        if not question:
            print("Please enter a question.")
            return

        print("Searching documents and generating answer...\n")

        try:
            answer, docs_found = ask_10k_question(question)

            display(Markdown("### Answer"))
            print(answer)

            display(Markdown("### Retrieved Sources"))
            for i, doc in enumerate(docs_found, 1):
                source = doc.metadata.get("source_file", doc.metadata.get("source", "Unknown source"))
                snippet = doc.page_content[:500].replace("\n", " ")
                print(f"{i}. {source}")
                print(f"   {snippet}...\n")

        except Exception as e:
            print("Error:", e)

ask_button.on_click(on_ask_clicked)

display(question_box)
display(ask_button)
display(output)

## Interactive 10-K Chatbot

Textarea(value='', description='Question:', layout=Layout(height='100px', width='90%'), placeholder='Ask a que…

Button(button_style='primary', description='Ask', icon='search', style=ButtonStyle())

Output()